# 02 · Libraries, modules, scripts

Where imported code comes from, how dependencies are pinned and reproduced, moving your code out of
cells into a module and a script, why notebooks and scripts behave differently, using every core,
and choosing between pandas and polars.

Some steps happen in a **terminal** at the repo root (`cd ~/code/netlab`), not in the notebook.
They're marked **Terminal**. Where it's harmless, the cell also shows a `!command` form that runs the
same thing from here.

> **Practice first:** finish notebook 01 (at least Exercises 1–3 and section G) before this one.
> Section D moves your notebook 01 functions into a file, so they need to exist. Where a section
> leans on a specific skill, its warm-up in `notebooks/practice/` is named at the top of it.

In [ ]:
import sys, os, time
from pathlib import Path
from netlab.check import check
from netlab import paths

def best_of(fn, repeat=3):
    runs = []
    for _ in range(repeat):
        t0 = time.perf_counter(); fn(); runs.append(time.perf_counter() - t0)
    return min(runs)

## A. Where imported code comes from

In 01 you saw that pandas is a folder on your disk. So is everything else you import.

In [ ]:
import json
print(json.__file__)
print(Path(json.__file__).read_text()[:600])    # the first lines of the json module: plain Python

In [ ]:
for i, folder in enumerate(sys.path):
    print(i, folder)

`sys.path` is the search list, in order. `import json` checks each folder for `json` and takes the
**first match**, the same way your shell searches the folders in `$PATH` for a command. A file
named `json.py` in an earlier folder would hide the real one.

In [ ]:
print("json" in sys.modules)          # already loaded?
import json as json_again
print(json_again is json)             # the same object: the second import didn't re-run the file
print(len(sys.modules), "modules loaded in this kernel right now")

### Exercise 1 — a prediction

The next cell writes a tiny module into a scratch folder. When imported, it prints a line. The cell
after imports it **three times**. Before running it, write down how many lines will print:

*Your prediction:* ___

In [ ]:
import tempfile
SCRATCH = Path(tempfile.gettempdir()) / "netlab_scratch"
SCRATCH.mkdir(exist_ok=True)
sys.path.insert(0, str(SCRATCH))     # put the scratch folder first in the search list
sys.modules.pop("beacon", None)      # forget any earlier import, so this demo starts clean
print(SCRATCH)

In [ ]:
%%writefile {SCRATCH}/beacon.py
print("beacon: module code is running")
INTERVAL = 30

In [ ]:
import beacon
import beacon
import beacon
print("INTERVAL =", beacon.INTERVAL)

One line. The first import runs the file and caches the module in `sys.modules`; the next two find
it in the cache and do nothing. (Run that cell again: zero lines.) `importlib.reload(beacon)` forces
a re-run; you'll rarely need it.

In [ ]:
del json_again
import json                 # creates the name `json`, the module
from json import dumps      # creates the name `dumps`, one function from inside it
import json as j            # creates the name `j`, the same module under another name

print(json is j, dumps is json.dumps)
print(dumps({"item": "apple", "qty": 3}))

## B. The environment and its dependencies

In [ ]:
print(sys.executable)
print(sys.prefix)

The Python running this notebook (the **kernel**) is inside this repo's `.venv/`. It's a separate
process using this project's private set of packages. That's why each project has its own: two
projects can need different versions of the same library without fighting.

**Terminal.** Run this in a terminal at the repo root. It adds three libraries this notebook needs, then come back.

```bash
uv add polars pyarrow requests
```

In [ ]:
!git diff ../pyproject.toml

In [ ]:
!grep -A 3 '^name = "polars"' ../uv.lock

In [ ]:
!uv tree --depth 2 | head -40

`pyproject.toml` lists what *you* asked for. `uv tree` shows what you got: pandas pulled in `numpy`,
`python-dateutil` and others you never asked for. Those are **transitive dependencies**, and
`uv.lock` pins every one of them to an exact version and a hash.

**Terminal.** Rebuild from nothing. First shut this notebook's kernel down (or close Jupyter); then in a terminal at the repo root:

```bash
rm -rf .venv
uv sync
```

Same environment back, in seconds. This is why `.venv/` is gitignored and `uv.lock` is committed:
the lockfile is the recipe, the venv is the meal. Anyone with the repo gets exactly your packages.

### Why pinning matters

1. **Reproducibility.** "Works on my machine" is almost always a version difference.
2. **Transitive drift.** You pinned pandas, but not what pandas depends on. The lockfile pins everything.
3. **Supply-chain security.** The lockfile records hashes, so a tampered or newly published malicious
   version can't slip in silently. Typosquatting is real: `uv add requets` would install a stranger's code.
4. **Upgrades are deliberate.** `uv lock --upgrade-package pandas` produces a reviewable diff.

In [ ]:
!cd .. && uv export --format requirements-txt --no-dev -q -o requirements.txt && head -12 requirements.txt && rm requirements.txt

`requirements.txt` is the older, flat format: still used by Docker images, deploy platforms and
tools that predate `pyproject.toml`. Note the `--hash` lines. We generated it, looked, and deleted
it: in this repo `uv.lock` is the source of truth.

### Updating a package that's already imported

Install an older version of a small library, `humanize`, import it, then upgrade it while the kernel
is running. In a terminal you'd run `uv add "humanize==4.16.0"`; it's harmless from here.

In [ ]:
!uv add -q "humanize==4.16.0"

In [ ]:
import humanize
print(humanize.__version__, "|", humanize.naturalsize(1_532_000))

In [ ]:
!uv add -q "humanize==4.17.0"

In [ ]:
import humanize
print("this kernel sees:   ", humanize.__version__)    # still the old one: sys.modules
!uv run python -c "import humanize; print('a fresh process sees:', humanize.__version__)"

The files on disk are new; this kernel still holds the old module in `sys.modules`. **Rule: after
changing installed packages, restart the kernel** (Kernel → Restart). Restart now, then run the next
two cells: the first re-imports, the second removes `humanize` (we don't need it).

In [ ]:
import humanize
print("after a restart:", humanize.__version__, "(if this is still the old version, you didn't restart)")

In [ ]:
!uv remove -q humanize

<details><summary>What people call this</summary>

A <code>.py</code> file you import is a <b>module</b>; a folder of them is a <b>package</b>. <code>sys.path</code> is the <b>module search path</b>.<br>
<code>.venv/</code> is a <b>virtual environment</b>. Libraries you use are <b>dependencies</b>; theirs are <b>transitive dependencies</b>.<br>
<code>uv.lock</code> is a <b>lockfile</b>: exact versions with hashes. Pinning protects against <b>supply-chain attacks</b> like <b>typosquatting</b>.
</details>

## C. Data in, data out

> **Practice first:** notebook 01 §G (your first look at pandas) and
> `practice/05_reshaping_data.ipynb` (rows vs columns). A pandas table is the column layout.

### A mock dataset, generated locally

100,000 shop orders from a seeded random generator: random-looking, but identical every run. A few
customers buy a lot and most buy a little, about 1% of orders are missing their total, and store
IDs have leading zeros. The generator uses NumPy (which came with pandas) to make millions of rows fast.

In [ ]:
import numpy as np
import pandas as pd

ITEMS = ["apple", "bread", "milk", "coffee", "cheese", "eggs", "tea", "rice", "pasta", "butter", "jam", "honey"]
ITEM_WEIGHTS = [15, 14, 13, 10, 6, 9, 5, 6, 7, 6, 4, 5]
PRICES = [50, 325, 199, 875, 640, 420, 350, 280, 230, 390, 450, 760]   # cents, same order as ITEMS
COUNTRIES = ["US", "GB", "DE", "FR", "CA", "JP", "AU", "BR", "IN", "NA"]   # "NA" is Namibia
COUNTRY_WEIGHTS = [30, 12, 10, 9, 8, 8, 7, 6, 6, 4]


def generate_orders(n: int, seed: int = 0) -> pd.DataFrame:
    """n mock shop orders. The same seed always gives the same orders."""
    rng = np.random.default_rng(seed)  # a seeded generator: "random", but reproducible
    customers = np.array([f"cust-{i:05d}" for i in range(1, 2001)])
    weights = rng.pareto(1.2, len(customers)) + 0.01  # a few customers buy a lot, most buy a little
    which = rng.choice(len(ITEMS), n, p=np.array(ITEM_WEIGHTS) / sum(ITEM_WEIGHTS))
    qty = rng.geometric(0.45, n)  # mostly 1 or 2, sometimes more
    price = np.array(PRICES)[which]
    amount = pd.array(qty * price, dtype="Int64")
    amount[rng.random(n) < 0.01] = pd.NA  # about 1% of orders lost their total
    stores = np.array([f"{i:04d}" for i in range(1, 121)])  # store IDs with leading zeros: "0042"
    seconds = np.sort(rng.integers(0, 7 * 24 * 3600, n))  # one week of orders, in time order
    return pd.DataFrame({
        "ts": np.datetime64("2026-03-01T00:00:00") + seconds.astype("timedelta64[s]"),
        "customer": rng.choice(customers, n, p=weights / weights.sum()),
        "item": np.array(ITEMS)[which],
        "qty": qty,
        "price_cents": price,
        "amount_cents": amount,
        "status": np.where(rng.random(n) < 0.9, "PAID", "REFUNDED"),
        "store_id": rng.choice(stores, n),
        "country_code": rng.choice(COUNTRIES, n, p=np.array(COUNTRY_WEIGHTS) / sum(COUNTRY_WEIGHTS)),
    })


orders = generate_orders(100_000, seed=42)
orders.head()

In [ ]:
paths.ensure_dirs()
csv_path = paths.DATA_RAW / "orders.csv"
orders.to_csv(csv_path, index=False)
print(csv_path, f"{csv_path.stat().st_size:,} bytes")

In [ ]:
import pandas as pd

back = pd.read_csv(csv_path)
pd.DataFrame({"before (in memory)": orders.dtypes.astype(str), "after (read from CSV)": back.dtypes.astype(str)})

In [ ]:
print(orders["store_id"].head(3).tolist(), "->", back["store_id"].head(3).tolist())
print(orders["amount_cents"].head(3).tolist(), "->", back["amount_cents"].head(3).tolist())
print(orders["country_code"].isna().sum(), "->", back["country_code"].isna().sum(), "blank country codes")

The round trip through CSV changed your data:
- timestamps became plain text;
- `amount_cents` became a float (CSV has no way to say "missing whole number", so pandas used a
  float, which can hold `NaN`, "not a number");
- `store_id` lost its leading zeros ("0042" became 42);
- every order from Namibia lost its country: its code is `"NA"`, and pandas reads the text `NA`
  as "missing" unless you tell it not to.

CSV carries text, not types. Notebook 03 §G fixes this.

Real data often arrives as lines of text, some of them broken. Write the same orders in the notebook 01 order-log format, with a few damaged lines mixed in:

In [ ]:
import random

def to_log_line(row):
    price = "<NA>" if pd.isna(row.amount_cents) else f"{row.price_cents / 100:.2f}"
    return f"{row.ts:%Y-%m-%d %H:%M} {row.customer} {row.item} {row.qty} {price}"

rng = random.Random(42)
lines = [to_log_line(row) for row in orders.head(2000).itertuples()]
for i in rng.sample(range(len(lines)), 10):
    lines[i] = lines[i][: rng.randint(5, 30)]          # cut off mid-line, as happens when a program crashes while writing
log_path = paths.DATA_RAW / "orders.log"
log_path.write_text("\n".join(lines) + "\n")
print(lines[0])
print(sum("<NA>" in l for l in lines), "lines with a missing price,", 10, "cut-off lines")

### A real dataset, downloaded

A public table of every country, with its two-letter code, name, region and currency, kept on
GitHub by the Open Knowledge Foundation's "datasets" project. Download it with `requests`, a
library for making web requests (notebook 03 goes deeper). Skip the download if the file is
already here, the same way you'd keep a downloaded installer instead of fetching it every time.

In [ ]:
import hashlib, json
from datetime import datetime, timezone
import requests

COUNTRIES_URL = "https://raw.githubusercontent.com/datasets/country-codes/main/data/country-codes.csv"
countries_path = paths.DATA_RAW / "country-codes.csv"
provenance_path = countries_path.with_suffix(".provenance.json")

if countries_path.exists():
    print("already downloaded; using the local copy")
else:
    response = requests.get(COUNTRIES_URL, timeout=60)
    response.raise_for_status()
    countries_path.write_bytes(response.content)
    provenance = {
        "url": COUNTRIES_URL,
        "retrieved_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "bytes": len(response.content),
        "sha256": hashlib.sha256(response.content).hexdigest(),
    }
    provenance_path.write_text(json.dumps(provenance, indent=2))
    print("downloaded")
print(provenance_path.read_text())

The provenance file is a receipt for data: where it came from, when, and a hash (a fingerprint
of the exact bytes) that proves the file you analyzed is the file you downloaded. If the source
changes tomorrow, you can tell.

In [ ]:
registry = pd.read_csv(countries_path)
print(registry.shape, "(rows, columns)")
registry[["ISO3166-1-Alpha-2", "CLDR display name", "Region Name", "Capital", "ISO4217-currency_alphabetic_code"]].head(8)

In [ ]:
code_col = registry["ISO3166-1-Alpha-2"]
print("blank country code:", code_col.isna().sum(), "->", registry.loc[code_col.isna(), "CLDR display name"].tolist())
print("blank region:      ", registry["Region Name"].isna().sum(), "->", registry.loc[registry["Region Name"].isna(), "CLDR display name"].tolist())
print("continent values:  ", registry["Continent"].value_counts(dropna=False).to_dict())

Real data, real problems. Namibia's code is `"NA"`, and pandas turned it into "missing". The
`Continent` column has the same problem: North America is `NA` there, and became `NaN` too.
Antarctica genuinely has no region. Read the file again and tell pandas that only an **empty**
cell means missing:

In [ ]:
registry = pd.read_csv(countries_path, keep_default_na=False, na_values=[""])
print("blank country code now:", registry["ISO3166-1-Alpha-2"].isna().sum())

countries = (
    registry
    .rename(columns={"ISO3166-1-Alpha-2": "code", "CLDR display name": "country",
                     "Region Name": "region", "ISO4217-currency_alphabetic_code": "currency"})
    [["code", "country", "region", "currency"]]
    .dropna(subset=["code"])                       # a row with no code can't be looked up: drop it
    .assign(region=lambda d: d["region"].fillna("(none)"))
    .drop_duplicates("code")
)
print(len(countries), "clean code -> country rows")
countries[countries["code"].isin(["US", "NA", "JP"])]

In [ ]:
joined = orders.merge(countries, left_on="country_code", right_on="code", how="left")
print("orders with no matching country:", joined["country"].isna().sum())
joined.groupby("region")["amount_cents"].sum().sort_values(ascending=False)

That's a **join**: for each order, look up its country code in the countries table, like a dict
lookup for every row at once, then a group-by to total the money per region.

## D. Your first module

> **Practice first:** notebook 01 Exercises 1–3 (`price_to_cents`, `cents_to_price`,
> `parse_order`). You'll paste your own versions into a file here. If you need a refresher,
> redo `practice/02_numbers_and_text.ipynb` and `practice/03_text_and_errors.ipynb`.

So far your functions live in notebook cells. Move them into a file in `src/netlab/`, so any
notebook, script or test can import them. First, make edited files reload automatically:

In [ ]:
%load_ext autoreload
%autoreload 2

`%autoreload 2` re-imports changed modules before each cell runs, so edits to `.py` files show up
without restarting the kernel. (That's the exception to §B's rule: it works for your own code.)

**Paste your own** `price_to_cents`, `cents_to_price` and `parse_order` from notebook 01 into the
next cell, replacing the stubs, then run it. If you haven't solved them yet, run it as-is: the checks
stay `NOT YET`.

In [ ]:
%%writefile ../src/netlab/parsing.py
"""Parsing helpers, moved here from notebook 01."""


def price_to_cents(price: str) -> int:
    raise NotImplementedError("your turn")


def cents_to_price(cents: int) -> str:
    raise NotImplementedError("your turn")


def parse_order(line: str) -> dict | None:
    raise NotImplementedError("your turn")

In [ ]:
from netlab.parsing import cents_to_price, parse_order, price_to_cents

check(parse_order, [
    (("2026-03-01 09:02 alice apple 3 0.50",),
     {"date": "2026-03-01", "time": "09:02", "customer": "alice", "item": "apple", "qty": 3, "price_cents": 50}),
    (("garbage",), None),
])
check(price_to_cents, [(("12.50",), 1250), (("7",), 700)])
check(cents_to_price, [((1250,), "12.50"), ((5,), "0.05")]);

In [ ]:
try:
    parsed = [parse_order(l) for l in log_path.read_text().splitlines()]
    good = [p for p in parsed if p is not None]
    print(f"{len(good)} parsed, {len(parsed) - len(good)} rejected out of {len(parsed)} lines")
except NotImplementedError:
    print("NOT YET (parse_order is still a stub)")

The rejected lines are the cut-off ones plus those with `<NA>` for a price: your function's
"return None for a line you can't parse" rule, doing its job on realistic data.

**Why does `from netlab.parsing import ...` work?** `src/` isn't on `sys.path` by default. The project
is installed into `.venv` in **editable** mode: a one-line `.pth` file points Python at `src/`.

In [ ]:
import site
site_packages = Path(site.getsitepackages()[0])
for pth in site_packages.glob("*.pth"):
    print(pth.name, "->", pth.read_text().strip())

That's how `netlab.check` has worked since notebook 01. Read the two modules that were already here:
`check.py` (the exercise checker: a small test runner) and `paths.py`.

In [ ]:
print((paths.ROOT / "src/netlab/check.py").read_text())

In [ ]:
print((paths.ROOT / "src/netlab/paths.py").read_text())

Now open `src/netlab/parsing.py` in your editor, change something visible (say, add a `print` at the
top of `parse_order`), save, and re-run the check cell above. Autoreload picks up the change. Undo it
when you're done.

### Your checks become tests

`check()` was a test runner in disguise. `pytest` is the standard one: it finds files named
`test_*.py`, runs every function named `test_*`, and reports each `assert` that fails. It runs
without a notebook, so it can run on every commit.

In [ ]:
%%writefile ../tests/test_parsing.py
from netlab.parsing import cents_to_price, parse_order, price_to_cents


def test_price_round_trip():
    assert price_to_cents("12.50") == 1250
    assert cents_to_price(1250) == "12.50"


def test_price_edges():
    assert price_to_cents("0.05") == 5
    assert price_to_cents("7") == 700
    assert cents_to_price(5) == "0.05"


def test_parse_order():
    row = parse_order("2026-03-01 09:02 alice apple 3 0.50")
    assert row["qty"] == 3 and row["price_cents"] == 50


def test_parse_order_rejects_garbage():
    assert parse_order("not an order line") is None

**Terminal.** In a terminal at the repo root (or run the cell below):

```bash
uv run pytest
```

In [ ]:
!cd .. && uv run pytest -q

If your parsing functions are still stubs, these tests fail with `NotImplementedError`. That's correct: a test's job is to fail until the code is right.

## E. Notebooks and scripts are different machines

### A notebook is a JSON file

In [ ]:
nb_path = paths.ROOT / "notebooks" / "02_libraries_modules_scripts.ipynb"
nb = json.loads(nb_path.read_text())
print(list(nb.keys()))
print(len(nb["cells"]), "cells")
print("first cell source:", "".join(nb["cells"][0]["source"])[:120], "...")
with_output = [cell for cell in nb["cells"] if cell.get("outputs")]
if with_output:
    print("a saved output:", json.dumps(with_output[0]["outputs"][0])[:300])
else:
    print("no outputs saved in the file yet (save the notebook, then re-run this cell)")

**Outputs are stored in the file.** Anything a cell ever printed — a token, a customer record, a
password from a misconfigured `print` — sits in the `.ipynb` and gets committed with it. That's why
this repo strips outputs on commit with `nbstripout`. It was installed when the repo was built; this
is the command, if you ever set up a repo yourself:

**Terminal.** In a terminal at the repo root (already done for this repo):

```bash
uv run nbstripout --install
```

In [ ]:
!cd .. && git check-attr filter -- notebooks/02_libraries_modules_scripts.ipynb
import subprocess
stripped = subprocess.run(["nbstripout", "-t", str(nb_path)], capture_output=True, text=True).stdout
kept = sum(len(cell.get("outputs", [])) for cell in json.loads(stripped)["cells"])
print("outputs in the file on disk:", sum(len(cell.get("outputs", [])) for cell in nb["cells"]))
print("outputs in what git would commit:", kept)

### Hidden state

Run the next cell, then **delete it** (select it, press `Esc` then `D D`), then run the one after.

In [ ]:
shop_name = "Corner Shop"

In [ ]:
print("shop_name is still", shop_name)

The cell is gone but the variable isn't. The kernel is one long-lived process with memory; the file
is not a faithful record of what ran. Look at the `[n]` execution counts on the left: they're
rarely in order. Anyone (including you, tomorrow) who runs the file top to bottom may get a
different result. **Kernel → Restart & Run All** is the honest test.

### A script is a fresh process, top to bottom, every time

Every Python file has a `__name__`. When it's imported, `__name__` is the module's name. When it's
run directly as a program, `__name__` is `"__main__"`.

In [ ]:
%%writefile {SCRATCH}/whoami.py
print("my __name__ is", repr(__name__))

if __name__ == "__main__":
    print("I was run as a program, so I do the program's work here")

In [ ]:
import whoami                          # imported: __name__ is "whoami"
!uv run python {SCRATCH}/whoami.py

`if __name__ == "__main__":` is how one file can be both: a module others import (definitions only),
and a program you run (the code under the `if`).

### Build your script

Move the order generator from §C into `src/netlab/orders.py`, plus a `main()` that reads command-line
arguments with `argparse`. The generator is the same code as §C, moved; `main()` is new.

In [ ]:
%%writefile ../src/netlab/orders.py
"""Mock order generator (moved from notebook 02 §C) and its command-line entry point."""

import argparse
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from netlab import paths

ITEMS = ["apple", "bread", "milk", "coffee", "cheese", "eggs", "tea", "rice", "pasta", "butter", "jam", "honey"]
ITEM_WEIGHTS = [15, 14, 13, 10, 6, 9, 5, 6, 7, 6, 4, 5]
PRICES = [50, 325, 199, 875, 640, 420, 350, 280, 230, 390, 450, 760]   # cents, same order as ITEMS
COUNTRIES = ["US", "GB", "DE", "FR", "CA", "JP", "AU", "BR", "IN", "NA"]   # "NA" is Namibia
COUNTRY_WEIGHTS = [30, 12, 10, 9, 8, 8, 7, 6, 6, 4]


def generate_orders(n: int, seed: int = 0) -> pd.DataFrame:
    """n mock shop orders. The same seed always gives the same orders."""
    rng = np.random.default_rng(seed)  # a seeded generator: "random", but reproducible
    customers = np.array([f"cust-{i:05d}" for i in range(1, 2001)])
    weights = rng.pareto(1.2, len(customers)) + 0.01  # a few customers buy a lot, most buy a little
    which = rng.choice(len(ITEMS), n, p=np.array(ITEM_WEIGHTS) / sum(ITEM_WEIGHTS))
    qty = rng.geometric(0.45, n)  # mostly 1 or 2, sometimes more
    price = np.array(PRICES)[which]
    amount = pd.array(qty * price, dtype="Int64")
    amount[rng.random(n) < 0.01] = pd.NA  # about 1% of orders lost their total
    stores = np.array([f"{i:04d}" for i in range(1, 121)])  # store IDs with leading zeros: "0042"
    seconds = np.sort(rng.integers(0, 7 * 24 * 3600, n))  # one week of orders, in time order
    return pd.DataFrame({
        "ts": np.datetime64("2026-03-01T00:00:00") + seconds.astype("timedelta64[s]"),
        "customer": rng.choice(customers, n, p=weights / weights.sum()),
        "item": np.array(ITEMS)[which],
        "qty": qty,
        "price_cents": price,
        "amount_cents": amount,
        "status": np.where(rng.random(n) < 0.9, "PAID", "REFUNDED"),
        "store_id": rng.choice(stores, n),
        "country_code": rng.choice(COUNTRIES, n, p=np.array(COUNTRY_WEIGHTS) / sum(COUNTRY_WEIGHTS)),
    })


def main(argv: list[str] | None = None) -> int:
    """Command-line entry point: parse arguments, generate, write. Returns the exit code."""
    parser = argparse.ArgumentParser(description="Generate mock shop orders.")
    parser.add_argument("--rows", type=int, default=100_000, help="number of orders (default 100000)")
    parser.add_argument("--seed", type=int, default=0, help="random seed (default 0)")
    parser.add_argument("--out", default=str(paths.DATA_RAW / "orders.csv"),
                        help="output file, or - for standard output (csv only)")
    parser.add_argument("--format", choices=["csv", "parquet"],
                        help="file format (default: from the --out extension)")
    args = parser.parse_args(argv)  # on a bad argument this prints usage and exits with code 2

    df = generate_orders(args.rows, args.seed)
    fmt = args.format or ("parquet" if args.out.endswith(".parquet") else "csv")
    if args.out == "-":
        df.to_csv(sys.stdout, index=False)  # data to stdout, so it can be piped
        return 0
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "parquet":
        df.to_parquet(out, index=False)
    else:
        df.to_csv(out, index=False)
    print(f"wrote {len(df):,} rows to {out}", file=sys.stderr)  # messages to stderr, data to stdout
    return 0


if __name__ == "__main__":
    sys.exit(main())


And the script: `scripts/` holds thin entry points. All the logic stays in `src/`, where it can be imported and tested.

In [ ]:
%%writefile ../scripts/make_orders.py
import sys

from netlab.orders import main

sys.exit(main())

**Terminal.** In a terminal at the repo root. `echo $?` prints the exit code of the last command: 0 means success.

```bash
uv run python scripts/make_orders.py --rows 1000 --seed 7 --out data/raw/orders_small.csv
echo $?
```

In [ ]:
!cd .. && uv run python scripts/make_orders.py --rows 1000 --seed 7 --out data/raw/orders_small.csv; echo "exit code: $?"

In [ ]:
# This cell fails on purpose: --rows needs a number
!cd .. && uv run python scripts/make_orders.py --rows lots; echo "exit code: $?"

In [ ]:
!cd .. && uv run python scripts/make_orders.py --rows 5 --out - | head -3

A nonzero exit code is how a program tells whoever ran it (you, a shell script, a scheduler, CI)
that it failed. Data goes to standard output, so it can be piped (`| head -3`); messages go to
standard error, so they don't pollute the data.

### An entry point

Give the script a proper command name. **Edit `pyproject.toml`** in your editor and add this section:

```toml
[project.scripts]
make-orders = "netlab.orders:main"
```

**Terminal.** Then in a terminal at the repo root:

```bash
uv sync
uv run make-orders --help
```

In [ ]:
!cd .. && uv run make-orders --help

`uv sync` created a small `make-orders` executable in `.venv/bin/` that calls `netlab.orders:main`.
Now **edit `README.md`** yourself: add a row to the "How to run" table for `uv run make-orders`.
A repo's entry points should be obvious to the next person.

**Rule of thumb: notebooks to explore, modules to keep, scripts to run.**

## F. Using more than one core

> **Practice first:** `practice/01_functions.ipynb`. This section passes functions around as
> values (`pool.map(cpu_task, ...)`): the function's name without `()` is the function itself.

Two kinds of slow work. **CPU-bound**: the processor is busy the whole time, here computing
checksums over blocks of data in pure Python. **I/O-bound**: the program is mostly waiting, here a
0.2 s sleep standing in for waiting on a download or a database.

In [ ]:
import random
import time


def checksum(data: bytes) -> int:
    """A checksum: one number computed from a block of data, used to tell whether the data changed."""
    if len(data) % 2:
        data += b"\x00"
    total = 0
    for i in range(0, len(data), 2):
        total += (data[i] << 8) | data[i + 1]
        total = (total & 0xFFFF) + (total >> 16)  # fold the carry back in
    return ~total & 0xFFFF


def cpu_task(seed: int, blocks: int = 6000, size: int = 1500) -> int:
    """Checksum `blocks` random blocks of data. Pure Python: all computing, no waiting."""
    rng = random.Random(seed)
    return sum(checksum(rng.randbytes(size)) for _ in range(blocks))


def io_task(i: int) -> int:
    """Stand-in for waiting on a download or a database: all waiting, no computing."""
    time.sleep(0.2)
    return i


t = best_of(lambda: cpu_task(1), repeat=1)
print(f"one cpu_task: {t:.2f} s; one io_task: 0.20 s")

In [ ]:
import multiprocessing, os
print("start method:", multiprocessing.get_start_method())
print("CPU cores:", os.cpu_count())

In [ ]:
# This cell fails on purpose (on this machine's start method)
from concurrent.futures import ProcessPoolExecutor

pool = ProcessPoolExecutor(max_workers=2)
try:
    print(list(pool.map(cpu_task, [1, 2], timeout=60)))
except Exception as e:
    print(type(e).__name__, "-", e)
finally:
    pool.shutdown(wait=False, cancel_futures=True)

On macOS (and Windows, and Linux from Python 3.14) new worker processes are started with **spawn**:
a brand-new Python that must *import* the function by name. `cpu_task` lives only in this kernel's
memory, not in any file, so each worker fails (`AttributeError: module '__main__' has no attribute
'cpu_task'`, in the red stderr text above) and the pool reports `BrokenProcessPool`.

**The fix:** put the functions in a module, a file workers can import.

In [ ]:
%%writefile ../src/netlab/work.py
"""CPU-bound and I/O-bound work for notebook 02 §F."""

import random
import time


def checksum(data: bytes) -> int:
    """A checksum: one number computed from a block of data, used to tell whether the data changed."""
    if len(data) % 2:
        data += b"\x00"
    total = 0
    for i in range(0, len(data), 2):
        total += (data[i] << 8) | data[i + 1]
        total = (total & 0xFFFF) + (total >> 16)  # fold the carry back in
    return ~total & 0xFFFF


def cpu_task(seed: int, blocks: int = 6000, size: int = 1500) -> int:
    """Checksum `blocks` random blocks of data. Pure Python: all computing, no waiting."""
    rng = random.Random(seed)
    return sum(checksum(rng.randbytes(size)) for _ in range(blocks))


def io_task(i: int) -> int:
    """Stand-in for waiting on a download or a database: all waiting, no computing."""
    time.sleep(0.2)
    return i


COUNTER = 0  # module-level: each process has its own copy


def bump(_):
    """Add one to this process's COUNTER and return it."""
    global COUNTER
    COUNTER += 1
    return COUNTER


In [ ]:
from netlab import work

with ProcessPoolExecutor(max_workers=2) as pool:
    print(list(pool.map(work.cpu_task, [1, 2])))

This is a concrete reason code graduates from notebooks into modules. Now measure all six combinations, with one worker per core:

In [ ]:
from concurrent.futures import ThreadPoolExecutor

WORKERS = os.cpu_count()

def run(kind, fn, n=WORKERS):
    t0 = time.perf_counter()
    if kind == "sequential":
        [fn(i) for i in range(n)]
    else:
        Pool = ThreadPoolExecutor if kind == "threads" else ProcessPoolExecutor
        with Pool(max_workers=WORKERS) as pool:
            list(pool.map(fn, range(n)))
    return time.perf_counter() - t0

grid = pd.DataFrame(
    {task: {kind: run(kind, fn) for kind in ["sequential", "threads", "processes"]}
     for task, fn in [("CPU task", work.cpu_task), ("I/O task", work.io_task)]}
).round(2)
print(f"{WORKERS} tasks each, {WORKERS} workers; seconds:")
grid

One Python process runs Python code on one core at a time (the **GIL**, global interpreter lock).
So threads don't speed up the CPU task. Threads *do* help the I/O task: while one waits, another
runs. Processes get separate interpreters and separate memory, so they help CPU work, but they cost
startup time and copying data between processes.

In [ ]:
work.COUNTER = 0
with ProcessPoolExecutor(max_workers=4) as pool:
    print("returned by worker processes:", list(pool.map(work.bump, range(8))))
print("parent's COUNTER after processes:", work.COUNTER)

with ThreadPoolExecutor(max_workers=4) as pool:
    list(pool.map(work.bump, range(8)))
print("parent's COUNTER after threads:  ", work.COUNTER)

Processes changed their own copies; the parent's counter didn't move. Threads share one memory, so
they changed the real one. Separate memory is why processes are safe from each other, and why
sharing results between them takes explicit copying. (Free-threaded Python builds, 3.13+, remove the
GIL and change this picture; out of scope here.)

<details><summary>What people call this</summary>

Busy computing: <b>CPU-bound</b>. Mostly waiting: <b>I/O-bound</b>.<br>
The one-at-a-time rule is the <b>GIL</b> (global interpreter lock). Threads give <b>concurrency</b>; processes give <b>parallelism</b>.<br>
Sending a function or data to another process means <b>pickling</b> (serializing) it.
</details>

## G. pandas vs polars

Generate a big file with your own command: 50,000,000 rows. (That count was chosen when this repo
was built so that pandas takes at least a couple of seconds on this machine.) It takes a while
the first time; the cell skips it if the file exists.

In [ ]:
big_path = paths.DATA_RAW / "orders_big.parquet"
if not big_path.exists():
    !cd .. && uv run make-orders --rows 50000000 --format parquet --out data/raw/orders_big.parquet
print(big_path, f"{big_path.stat().st_size / 1e6:,.0f} MB")

> **Practice first:** `practice/07_totals_and_sorting.ipynb`. The question below is notebook 01
> Exercise 9 (`top_customers`) at 50 million rows.

Same question three ways: money per `customer`, `PAID` orders only, top 10. Watch CPU usage in Activity
Monitor (or `htop`) while each runs: one core busy, or all of them? (Callback to §F.)

In [ ]:
import polars as pl

def with_pandas():
    df = pd.read_parquet(big_path)
    return df[df["status"] == "PAID"].groupby("customer")["amount_cents"].sum().nlargest(10)

def with_polars_eager():
    df = pl.read_parquet(big_path)
    return (df.filter(pl.col("status") == "PAID").group_by("customer")
              .agg(pl.col("amount_cents").sum()).sort("amount_cents", descending=True).head(10))

def polars_lazy_query():
    return (pl.scan_parquet(big_path).filter(pl.col("status") == "PAID").group_by("customer")
              .agg(pl.col("amount_cents").sum()).sort("amount_cents", descending=True).head(10))

timings = {   # polars first: see the note below the numbers
    "polars lazy": best_of(lambda: polars_lazy_query().collect(), repeat=2),
    "polars eager": best_of(with_polars_eager, repeat=2),
    "pandas": best_of(with_pandas, repeat=2),
}
for name, seconds in timings.items():
    print(f"{name:>13}: {seconds:6.2f} s")

**A measuring lesson from building this notebook:** timed in the other order, the polars run right
after pandas came out about 5x slower on this machine, while the memory pandas had just released was
being handed back to the OS. Order and leftover state can distort a benchmark. Try swapping the order.

In [ ]:
print(with_pandas().head(3))
print(polars_lazy_query().collect().head(3))

In [ ]:
print(polars_lazy_query().explain())

Read the plan bottom-up (like a traceback). The Parquet scan already contains the filter
(`SELECTION`) and reads only the columns the query uses (`PROJECT 3/9 COLUMNS`): the filter and
column choice are **pushed down** into the file read, so most of the file is never loaded.

Why polars is faster: columnar Arrow memory, every core working at once, and a query planner that
reads less data.

### The downsides, shown

**1. Strictness.** pandas quietly makes something of mixed data; polars refuses.

In [ ]:
mixed = pd.Series([1, "2", 3])
print("pandas makes a column of", mixed.dtype, "and a comparison quietly gives", (pd.Series([1, 2]) == "1").tolist())

In [ ]:
# This cell fails on purpose
for label, attempt in [("mixed column", lambda: pl.Series([1, "2", 3])),
                       ("int vs str", lambda: pl.DataFrame({"qty": [1, 3]}).filter(pl.col("qty") == "1"))]:
    try:
        attempt()
    except Exception as e:
        print(f"{label}: {type(e).__name__} - {str(e).splitlines()[0]}")

Strict is usually what you want in a pipeline (garbage in should fail loudly), but it means more explicit casting.

**2. No index.** pandas users reach for row labels; polars has none, so you filter instead.

In [ ]:
small_pd = orders.head(1000).set_index("customer")
some_customer = small_pd.index[0]
print(small_pd.loc[[some_customer], "amount_cents"].head(3).tolist())       # pandas: look up by row label

small_pl = pl.from_pandas(orders.head(1000))
print(small_pl.filter(pl.col("customer") == some_customer)["amount_cents"].head(3).to_list())   # polars: filter by value

In [ ]:
# This cell fails on purpose
try:
    small_pl.set_index("customer")
except Exception as e:
    print(type(e).__name__, "-", e)

**3. Ecosystem.** Many plotting and ML libraries expect pandas. Converting is possible, but measure its cost on the big frame:

In [ ]:
big_pl = pl.read_parquet(big_path)
print(f"{big_pl.height:,} rows: .to_pandas() takes {best_of(big_pl.to_pandas, repeat=1):.2f} s")
del big_pl

**4. API churn.** polars has changed its API across versions faster than pandas, so older tutorials
and AI-generated code are wrong more often. Example: `groupby` was renamed `group_by`.

In [ ]:
# This cell fails on purpose
print("polars", pl.__version__)
try:
    small_pl.groupby("customer")
except Exception as e:
    print(type(e).__name__, "-", e)

Your version is pinned in `uv.lock`; check the docs for *that* version, not the newest.

**Summary:** pandas for small data and ecosystem fit; polars for large data, pipelines, and when
speed matters. Both are the shapes from notebook 01, held as columns.

<details><summary>What people call this</summary>

Building the query first and running it later is <b>lazy evaluation</b>; the planner moving a filter into the read is <b>predicate pushdown</b> (and <b>projection pushdown</b> for columns).<br>
Arrow is a <b>columnar memory format</b> shared by polars, pyarrow and pandas.
</details>

## What you can do now

- Find where any import comes from, and know why a second import does nothing.
- Add, lock, rebuild and upgrade dependencies, and explain why the lockfile is committed.
- Download data with provenance, clean a messy real file, and join it to your own data.
- Move code from cells into a module, test it with pytest, and run it as a script with arguments
  and exit codes.
- Pick threads or processes by whether the work waits or computes, and measure it.
- Choose pandas or polars for a job, knowing what each costs.

## Glossary: what people call these

| Term | Meaning, in this notebook's terms |
|---|---|
| module | a `.py` file you import |
| package | a folder of modules (`netlab`) |
| virtual environment | `.venv/`: this project's private Python and libraries |
| dependency | a library your code imports |
| transitive dependency | a library your libraries import |
| lockfile | `uv.lock`: exact versions and hashes of everything |
| editable install | the `.pth` file pointing Python at `src/` |
| entry point | `make-orders`: a command name mapped to a function |
| kernel | the long-lived Python process behind a notebook |
| GIL | the rule that one process runs Python on one core at a time |
| CPU-bound | slow because it computes |
| I/O-bound | slow because it waits |
| lazy evaluation | build the whole query, then run it |
| predicate pushdown | the filter applied while reading the file, not after |

## Commit checkpoint

```bash
git status
git add pyproject.toml uv.lock README.md src/netlab/ scripts/ tests/ notebooks/02_libraries_modules_scripts.ipynb
git commit -m "notebook 02: parsing module, order generator script, tests"
```

Note what you're committing: code and the lockfile. Not `data/` (gitignored, regenerable), not
`.venv/` (rebuilt by `uv sync`). Run `git status` afterwards: nothing in `data/` should appear.